# 04b LLM pipeline 1: structured JSON to a natural language explanation

The LLM receives global feature importance and local SHAP/EBM contributions as
structured JSON and writes a German explanation for non technical users.

* Advantage: precise, machine readable input.
* Disadvantage: no visual context (no plots).

For each combination (XGBoost/EBM x 10 instances) one explanation is generated
and saved in results/pipeline04/.

In [1]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    INSTANCE_IDS,
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR,
)
from utils.llm import ask_text, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = "poisson_log"
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION

# n=20 validity run: the 10 instances, 1 generation, real time,
# output directly into results/pipeline04/
GEN_INSTANCE_IDS = INSTANCE_IDS
N_GEN            = 1
OUT_DIR          = RESULTS_DIR / "pipeline04"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM model:     {MODEL}")
print(f"Loss option:   {LOSS_KEY}")
print(f"Instances:     {len(GEN_INSTANCE_IDS)} x {N_GEN} generation")
print(f"Output:        {OUT_DIR}")

LLM model:     claude-sonnet-4-6
Loss option:   poisson_log
Instances:     10 x 1 generation
Output:        /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/pipeline04


## 1. System prompt

The system prompt is cached (Anthropic prompt caching). Since the minimum for a
cache block is 1024 tokens, the system prompt contains, besides the instructions,
the full feature schema with descriptions, so it reliably crosses the threshold and
is read from the cache on all later calls.

In [2]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_04_json.md").read_text()

print(f"System prompt: {len(SYSTEM_PROMPT)} characters, ~{len(SYSTEM_PROMPT)//4} tokens (estimated)")

System prompt: 6410 characters, ~1602 tokens (estimated)


## 2. Helper functions

build_context_string turns normalised raw values into readable values
(for example temp=0.68 to ~27.9 C). This context line is attached to the JSON
payload as human_readable_context so the model does not have to denormalise itself.

In [3]:
from utils.explanations import build_context_string
from utils import load_global_explanation, load_local_explanation

# load_global_explanation / load_local_explanation: central IO helpers
# (utils.generation), path and loss schema in one place, tested.
# build_user_prompt stays visible: it defines what the LLM receives as JSON.

def build_user_prompt(global_exp: dict, local_exp: dict, top_k: int = 5) -> str:
    fv = local_exp["feature_values"]
    payload = {
        "model": local_exp["model"],
        "metrics": {
            "rmse":             global_exp["metrics"]["rmse"],
            "r2":               global_exp["metrics"]["r2"],
            "poisson_deviance": global_exp["metrics"]["poisson_deviance"],
        },
        "global_top_features": [
            {"rank": e["rank"], "feature": e["feature"]}
            for e in global_exp["global_importance"][:top_k]
        ],
        "instance_id":           local_exp["instance_id"],
        "feature_values":        fv,
        "human_readable_context": build_context_string(fv),
        "y_true":                local_exp["y_true"],
        "prediction":            local_exp["prediction"],
        "top_contributions":     local_exp["contributions"][:6],
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# Show an example prompt
g = load_global_explanation("xgb", loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
l = load_local_explanation("xgb", INSTANCE_IDS[0], loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
sample_prompt = build_user_prompt(g, l)
print(f"Example user prompt ({len(sample_prompt)} characters):")
print(sample_prompt)

Example user prompt (1358 characters):
{
  "model": "xgb",
  "metrics": {
    "rmse": 45.439511,
    "r2": 0.935358,
    "poisson_deviance": 9.380072
  },
  "global_top_features": [
    {
      "rank": 1,
      "feature": "hr"
    },
    {
      "rank": 2,
      "feature": "yr"
    },
    {
      "rank": 3,
      "feature": "temp"
    },
    {
      "rank": 4,
      "feature": "weekday"
    },
    {
      "rank": 5,
      "feature": "mnth"
    }
  ],
  "instance_id": 224,
  "feature_values": {
    "weathersit": 1.0,
    "mnth": 2.0,
    "hr": 19.0,
    "weekday": 4.0,
    "yr": 0.0,
    "holiday": 0.0,
    "temp": 0.2,
    "hum": 0.4,
    "windspeed": 0.0
  },
  "human_readable_context": "19:00, Thursday, February, 2011, clear/few clouds, ~8.2 C, 40 % humidity, wind 0.0 km/h",
  "y_true": 96.0,
  "prediction": 111.9967,
  "top_contributions": [
    {
      "feature": "temp",
      "value": 0.2,
      "contribution": -0.548818
    },
    {
      "feature": "hr",
      "value": 19.0,
   

## 3. LLM calls

In [4]:
from utils import run_resumable_generation, build_generation_record
from utils.llm import build_text_params, run_params

# n=20 validity: real time over the central resume loop (skip if exists,
# idempotent, lossless, tested in tests/test_generation_loop.py). The persisted
# record is built via utils.build_generation_record (one schema for all
# pipelines, golden test in test_generation_loop.py).

def generate_json(model_name, iid, gen_idx):
    g_exp = load_global_explanation(model_name, loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
    l_exp = load_local_explanation(model_name, iid, loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
    params = build_text_params(
        build_user_prompt(g_exp, l_exp),
        system=SYSTEM_PROMPT, model=MODEL, max_tokens=MAX_TOKENS, cache_system=True,
    )
    t0 = time.time()
    response = run_params(params)
    elapsed  = time.time() - t0

    text  = strip_scratchpad(response["content"][0]["text"])
    usage = response.get("usage", {})
    record = build_generation_record(
        pipeline="04_json", model_name=model_name, instance_id=iid,
        explanation=text, usage=usage, llm_model=MODEL, loss_key=LOSS_KEY,
        prediction=l_exp["prediction"], y_true=l_exp["y_true"],
        elapsed_s=round(elapsed, 2),
    )
    u = record["usage"]
    print(f"  {model_name.upper()} inst={iid:4d} g{gen_idx}  "
          f"pred={record['prediction']:6.1f}  y={record['y_true']:5.0f}  "
          f"in={u['input_tokens']}  out={u['output_tokens']}  "
          f"cache={u['cache_read_input_tokens']}  t={elapsed:.1f}s")
    return record


results = run_resumable_generation(
    model_names=["xgb", "ebm"],
    instance_ids=GEN_INSTANCE_IDS,
    out_dir=OUT_DIR,
    generate=generate_json,
    n_generations=N_GEN,
)

totals = {
    "in":    sum(r["usage"]["input_tokens"] for r in results),
    "out":   sum(r["usage"]["output_tokens"] for r in results),
    "cache": sum(r["usage"]["cache_read_input_tokens"] for r in results),
}
print(f"\nTotal:  input={totals['in']}  output={totals['out']}  "
      f"cache_read={totals['cache']}  ({len(results)} units)")

  XGB inst= 224 g0  pred= 112.0  y=   96  in=598  out=502  cache=0  t=11.3s
  XGB inst= 580 g0  pred=  64.6  y=   63  in=599  out=515  cache=1776  t=11.1s
  XGB inst=1041 g0  pred= 385.4  y=  387  in=600  out=497  cache=1776  t=10.5s
  XGB inst=1481 g0  pred= 228.6  y=  277  in=600  out=545  cache=1776  t=11.5s
  XGB inst=1677 g0  pred= 254.3  y=  286  in=600  out=516  cache=1776  t=11.3s
  XGB inst=2058 g0  pred= 194.1  y=  243  in=599  out=569  cache=1776  t=12.6s
  XGB inst=2510 g0  pred= 326.0  y=  372  in=600  out=536  cache=1776  t=11.3s
  XGB inst=3543 g0  pred= 309.9  y=  286  in=601  out=508  cache=1776  t=11.5s
  XGB inst=3847 g0  pred= 585.8  y=  531  in=600  out=525  cache=1776  t=12.5s
  XGB inst=4454 g0  pred= 297.3  y=  354  in=602  out=563  cache=1776  t=12.9s
  EBM inst= 224 g0  pred= 117.0  y=   96  in=598  out=530  cache=1776  t=11.5s
  EBM inst= 580 g0  pred=  73.1  y=   63  in=598  out=497  cache=1776  t=11.4s
  EBM inst=1041 g0  pred= 389.7  y=  387  in=599  out=5

## 4. Example explanations

In [5]:
for rec in results[:2]:
    sep = '=' * 70
    print(sep)
    print(f"Model: {rec['xai_model'].upper()}  |  instance: {rec['instance_id']}  "
          f"|  prediction: {rec['prediction']:.1f}  |  actual: {rec['y_true']:.0f}")
    print(sep)
    print(rec["explanation"])
    print()

Model: XGB  |  instance: 224  |  prediction: 112.0  |  actual: 96
<prediction>The model predicted approximately 112 rented bikes for this hour, while the actual count was 96. That is an overestimate of about 17 percent, which is a moderate match — reasonable but not precise for this particular instance.</prediction>

<drivers>The strongest influence this hour is the cold temperature of about 8 °C (normalised value 0.2), which drags demand down noticeably (rank 1, influence −0.55). This is the single largest factor shaping the prediction. Partially offsetting that is the hour of day: 19:00 is right in the middle of the evening commuter peak, pushing demand up strongly (rank 2, influence +0.41). However, two further factors reinforce the cold-weather drag: February as a winter month adds a clear seasonal dampener (rank 3, influence −0.32), and the year 2011 (yr=0) acts as an additional damping factor because 2011 represented the lower-demand phase of the system's history (rank 4, influen

## 5. Summary

In [6]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Model":      r["xai_model"].upper(),
        "Instance":   r["instance_id"],
        "y_true":     r["y_true"],
        "Prediction": r["prediction"],
        "Words":      len(r["explanation"].split()),
        "tok_input":  r["usage"]["input_tokens"],
        "tok_output": r["usage"]["output_tokens"],
        "Time (s)":   r["elapsed_s"],
    }
    for r in results
])
display(summary)

,Model,Instance,y_true,Prediction,Words,tok_input,tok_output,Time (s)
0,XGB,224,96.0,111.9967,231,598,502,11.29
1,XGB,580,63.0,64.5773,234,599,515,11.11
2,XGB,1041,387.0,385.3899,239,600,497,10.50
3,XGB,1481,277.0,228.5813,254,600,545,11.54
4,XGB,1677,286.0,254.2623,247,600,516,11.35
5,XGB,2058,243.0,194.0592,279,599,569,12.62
6,XGB,2510,372.0,326.0351,256,600,536,11.29
7,XGB,3543,286.0,309.9225,246,601,508,11.54
8,XGB,3847,531.0,585.7666,262,600,525,12.48
9,XGB,4454,354.0,297.3312,276,602,563,12.88
